# Week 2: Tokenization and Inference 

This notebook usees the end-to-end transformer from last week, and abstracts most of it away to focus on tokenization and inference.

Last week we saw that the transformer was trained to reproduce Wikipedia text character-by-character. We could also have it predict word-by-word. These units of prediction in both cases are called **tokens**.

In practice, most modern language models use something in between: **subword tokens**. One common approach is Byte Pair Encoding (BPE), which builds a vocabulary of frequently occurring sequences of characters or bytes. This lets common words be represented by a single token, while less common words might be split into multiple tokens. [Look here](https://huggingface.co/spaces/Xenova/the-tokenizer-playground) to see how various models distinguish tokens.

Recall the overall transformer architecture from [Attention is All You Need](https://arxiv.org/pdf/1706.03762).

For now, consider the attention and FFN blocks as a "black box." What does this "black box" function do? It takes in a sequence of tokens (which the input text has to be split into) and outputs a set of scores, called **logits**, for what the *single next token* could be. We can turn these into a probability distribution and choose one out of every possible next token.

In a letter-for-letter model like our Wikipedia transformer, that looks like this:

- text: `"the transformer is"`
- Tokenize: `X = ['t', 'h', 'e', ' ', 't', 'r', 'a', 'n', 's', 'f', 'o', 'r', 'm', 'e', 'r', ' ', 'i', 's']`
  - Note that spaces are themselves tokens.
  - Technically, each token has a numerical ID. Before entering the transformer blocks, these IDs are converted into vectors called **embeddings**. We'll look at how this works later.
- Put the tokens through the model: `X -> transformer -> Y`
- `Y = {a: -1, b: -2.1, ..., ' ': 3}`
  - These are called logits. They are arbitrary scores which can go negative. Lower is less likely, higher is more likely.
  - Technically, the transformer produces logits at every position in the input sequence. For generation, we only care about the last position, which predicts the next token.
- Turn logits into probabilities: `softmax(Y) = {a: ..., b: ..., ' ': ...}`
  - To get probabilities, we apply a function called *softmax*. This converts the logits into probabilities that sum to 1.
  - [Softmax Demonstration](https://www.desmos.com/calculator/pmko7malg0)
- We can either choose the top probability every time (**greedy decoding**), or sample randomly according to the probability distribution.
  - We can also adjust the distribution using **temperature**. A temperature of 1 leaves the distribution unchanged, lower temperatures favor the most likely tokens, and higher temperatures make the distribution more random. Temperature 0 is typically used to mean greedy decoding.

And now, we can get the next token as, say, `sample(softmax(Y))`. Let's append it to X -- `X.append(sample(softmax(Y)))` -- and repeat this process.

```python
['t'] -> transformer -> 'h'
['t', 'h'] -> transformer -> 'e'
['t', 'h', 'e'] -> transformer -> ' '
['t', 'h', 'e', ' '] -> transformer -> 't'
['t', 'h', 'e', ' ', 't'] -> transformer -> 'r'
['t', 'h', 'e', ' ', 't', 'r'] -> transformer -> 'a'
...
```

This process is called **autoregressive generation**. We generate one token at a time, append it to the input, and use the updated sequence to predict the next token.

The important distinction is that the transformer itself doesn't actually choose the next token. It produces logits, and the **generation algorithm** decides what to do with them. By modifying how we select the next token, we can change the model's behavior without changing the transformer itself.

In [ ]:
import numpy as np
import re
import transformer_util as tu
from matplotlib import pyplot as plt

# TOKENIZATION_APPROACH = "ascii"
TOKENIZATION_APPROACH = "words"

with open("wikipedia_corpus.txt", encoding="utf-8") as file:
    corpus_text = file.read()

if TOKENIZATION_APPROACH == "ascii":
    ASCII_CHARS = "\n" + "".join(
        chr(code) for code in range(32, 127)
    )

    CHAR_TO_ID = {
        char: index
        for index, char in enumerate(ASCII_CHARS)
    }

    ID_TO_CHAR = {
        index: char
        for char, index in CHAR_TO_ID.items()
    }

    VOCAB_SIZE = len(ASCII_CHARS)
    UNKNOWN_ID = CHAR_TO_ID["?"]

    def encode(text):
        token_ids = [
            CHAR_TO_ID.get(char, UNKNOWN_ID)
            for char in text
        ]
        return np.asarray(token_ids, dtype=np.int64)


    def decode(token_ids):
        return "".join(
            ID_TO_CHAR[int(token_id)]
            for token_id in token_ids
        )

    tokens = encode(corpus_text)
elif TOKENIZATION_APPROACH == "words":
    # SPLIT to isolate alphabetic strings, digit strings, and punctuation
    word_split = lambda text: re.findall(r'[^\W\d_]+|[\s\S]', text)
    WORDS = word_split(corpus_text)

    WORD_TO_ID = {
        word: index
        for index, word in enumerate(set(WORDS))
    }

    ID_TO_WORD = {
        index: word
        for word, index in WORD_TO_ID.items()
    }

    VOCAB_SIZE = len(WORD_TO_ID)
    UNKNOWN_ID = WORD_TO_ID.get("?", 0)

    def encode(text):
        token_ids = [
            WORD_TO_ID.get(word, UNKNOWN_ID)
            for word in word_split(text)
        ]
        return np.asarray(token_ids, dtype=np.int64)

    def decode(token_ids):
        return " ".join(
            ID_TO_WORD[int(token_id)]
            for token_id in token_ids
        )

    tokens = encode(corpus_text)
else:
    raise ValueError("Unsupported tokenization approach")

print('Used tokenization scheme:', TOKENIZATION_APPROACH)
print('Vocabulary size:', VOCAB_SIZE)
print('Corpus length:', len(tokens))
print('Example tokens:')
for token in tokens[:15]:
    print("'", decode([token]), "'", end=", ", sep='')

# # uncomment to test arbitrary text tokenization
# for token in encode('the year 1999, (2003)'):
#     print("'", decode([token]), "'", end=", ", sep='')

In [ ]:
def get_batch_generator(tokens, batch_size, sequence_length, rng):
    def get_batch():
        window_length = sequence_length + 1
        max_start = len(tokens) - window_length

        if max_start < 0:
            raise ValueError(
                "Token sequence is shorter than sequence_length + 1"
            )

        starts = np.arange(0, max_start + 1, sequence_length)
        rng.shuffle(starts)

        offsets = np.arange(window_length)
        for i in range(0, len(starts) - batch_size + 1, batch_size):
            batch_starts = starts[i:i + batch_size]
            windows = tokens[batch_starts[:, None] + offsets]
            yield windows[:, :-1], windows[:, 1:]

    return get_batch

params = tu.initialize_parameters(
    vocab_size=VOCAB_SIZE,
    d_model=64,
    d_ff=256,
    n_layers=3,
    n_heads=4,
    seed=0,
)

# NOTE: saving this is necessary if training is resumed
adam_state = {
    "momentum": tu.zeros_like_tree(params),
    "squared_grad": tu.zeros_like_tree(params),
    "step": 0,
}

BATCH_SIZE = 8
SEQ_LENGTH = 64
rng = np.random.default_rng(0)
get_batch = get_batch_generator(
    tokens,
    batch_size=BATCH_SIZE,
    sequence_length=SEQ_LENGTH,
    rng=rng,
)

losses = [] # Define losses here so multiple training sessions can append to the same list

In [ ]:
# NOTE: you can run this cell multiple times to continue training the model

number_of_steps = 2000

for step in range(number_of_steps):
    inputs, targets = next(get_batch())

    logits, cache = tu.transformer(inputs, params)

    loss = tu.softmax_cross_entropy(logits, targets)

    grad_logits = tu.softmax_cross_entropy_backward(
        logits,
        targets,
    )

    grads = tu.transformer_backward(
        params,
        cache,
        grad_logits,
    )

    tu.adam_step(
        params,
        grads,
        adam_state,
        learning_rate=3e-4,
    )

    if step % 100 == 0:
        epochs = step/(len(tokens)//BATCH_SIZE)
        print(f"{step=:0{len(str(number_of_steps-1))}d}, {loss=:.4f}, {epochs=:.4f}")

    losses.append(loss)

plt.plot(losses)
plt.plot(np.convolve(losses, np.ones(100)/100, mode='valid'))
plt.xlabel("Iteration")
plt.ylabel("Loss")
plt.title("Training Loss")
plt.show()

tu.save_model(params, adam_state, encode, decode, f"{TOKENIZATION_APPROACH}-model.dill")

In [ ]:
generated_text = tu.generate(
    tokenizer=(encode, decode, VOCAB_SIZE),
    params=params,
    prompt="The transformer",
    number_of_tokens=100,
    context_length=SEQ_LENGTH,
    temperature=1.0,
    seed=100,
    SHOW_ATTENTION=True
)

# But what's actually going on behind the generation?

In [ ]:
context = encode("The transformer ")[np.newaxis, :] # newaxis is for batch dimension. Required even though we have just one token sequence

logits, cache = tu.transformer(context, params)

# NOTE: why are there 3 outputs? B/c the transformer predicts all positions' next tokens in parallel. We only need the last one
print("logits.shape =", logits.shape)
logits = logits[0,-1,:]
print("logits =", logits)
print('-'*50)

# Convert logits to probabilities using softmax
# https://www.desmos.com/calculator/pmko7malg0
probabilities = tu.softmax(logits)

# print top 10 most likely next tokens
top_10_indices = np.argsort(probabilities)[-10:][::-1]
top_10_probabilities = probabilities[top_10_indices]
for i, prob in zip(top_10_indices, top_10_probabilities):
    print(f'"{decode([i])+"\" (id = "+str(i)+")":<30} logit = {logits[i]:>8.4f} \t probability = {prob:>8.4f}')
print('-'*50)

# What does random sampling look like?
next_token = rng.choice(VOCAB_SIZE, p=probabilities)
print(f'Randomly sampled next token: "{decode([next_token])}" (id = {next_token})')
print(decode(context[0])+decode([next_token]))

# Let's use this to recreate the generation loop

And once we have our own generation loop, what are some variations on sampling?
- Temperature
- Repetition penalty.
  - Most API's support something like this. This can help the repetitive outputs we see when temperature is low
- [Top-p sampling](https://en.wikipedia.org/wiki/Top-p_sampling)
  - [Top-k sampling](https://en.wikipedia.org/wiki/Top-p_sampling#Top-k_sampling)
- Most LLM inference APIs let you [alter logits](https://help.openai.com/en/articles/5247780-using-logit-bias-to-alter-token-probability-with-the-openai-api) to encourage or supress certain tokens.
  - Note: this is a way to adjust LLM behavior without training or finetuning the weights! E.g., you could increase the tokens logits for words related to a topic.
- Some LLM inference APIs go as far as allowing a [grammar to drive token generation](https://github.com/ggml-org/llama.cpp/blob/master/grammars/README.md), where you specify some format (e.g., for a programming language, or for a data format like JSON) to force the LLM output.
- *Custom* sampling approaches?

In [ ]:
# ...